## Silver Layer — Activity Splits
Clean lap/split data for pace analysis and interval training insights.

**Purpose:** One row per lap with standardized metrics for analyzing pacing strategies and interval workouts.

**Transformations:**
* **Unit conversions:** meters → km, seconds → minutes, m/s → km/h
* **Cadence adjustment:** Double cadence values (Garmin reports per-foot, convert to steps per minute)
* **Column drops:** JSON blobs (lengthDTOs, connectIQMeasurement), lat/long coordinates, temperature fields, internal IDs
* **Standardization:** snake_case column names, consistent data types

**Key Metrics Preserved:**
* Distance, duration, elevation per lap
* Speed and pace metrics
* Heart rate and cadence
* Workout intensity type (ACTIVE/REST)

In [0]:
%sql
CREATE OR REPLACE TABLE garmin_lakehouse.silver.splits AS
SELECT
    -- Identifiers
    CAST(activityId AS STRING) as activity_id,
    lapIndex as lap_index,
    
    -- Timestamp
    CAST(startTimeGMT AS TIMESTAMP) as start_time_gmt,
    
    -- Distance (meters → km)
    ROUND(distance / 1000, 2) as distance_km,
    
    -- Duration (seconds → minutes)
    ROUND(duration / 60, 2) as duration_minutes,
    ROUND(movingDuration / 60, 2) as moving_duration_minutes,
    
    -- Elevation (keep in meters)
    ROUND(elevationGain, 1) as elevation_gain_m,
    ROUND(elevationLoss, 1) as elevation_loss_m,
    
    -- Pace (min/km) instead of speed - handle zero/null speed values
    ROUND(60 / NULLIF(averageSpeed * 3.6, 0), 2) as avg_pace_min_per_km,
    ROUND(60 / NULLIF(averageMovingSpeed * 3.6, 0), 2) as avg_moving_pace_min_per_km,
    ROUND(60 / NULLIF(maxSpeed * 3.6, 0), 2) as best_pace_min_per_km,
    
    -- Calories
    ROUND(calories, 0) as calories,
    
    -- Heart Rate
    ROUND(averageHR, 0) as avg_heart_rate,
    ROUND(maxHR, 0) as max_heart_rate,
    
    -- Cadence (keep as Garmin reports - per foot)
    ROUND(averageRunCadence, 0) as avg_cadence_rpm,
    ROUND(maxRunCadence, 0) as max_cadence_rpm,
    
    -- Stride length (cm → meters)
    ROUND(strideLength / 100, 2) as stride_length_m,
    
    -- Workout intensity
    intensityType as intensity_type,
    
    -- Metadata
    CURRENT_TIMESTAMP() as loaded_at
    
FROM garmin_lakehouse.raw.splits

In [0]:
%sql
SELECT 
    COUNT(*) as total_laps,
    COUNT(DISTINCT activity_id) as unique_activities,
    ROUND(AVG(distance_km), 2) as avg_lap_distance_km,
    ROUND(MIN(distance_km), 2) as min_lap_distance_km,
    ROUND(MAX(distance_km), 2) as max_lap_distance_km,
    ROUND(AVG(duration_minutes), 1) as avg_lap_duration_min,
    ROUND(AVG(avg_pace_min_per_km), 2) as avg_pace_min_per_km,
    ROUND(AVG(avg_heart_rate), 0) as avg_heart_rate,
    ROUND(AVG(avg_cadence_rpm), 0) as avg_cadence_rpm,
    COUNT(DISTINCT intensity_type) as unique_intensity_types
FROM garmin_lakehouse.silver.splits

In [0]:
%sql
SELECT 
    activity_id,
    lap_index,
    start_time_gmt,
    distance_km,
    duration_minutes,
    avg_pace_min_per_km,
    avg_heart_rate,
    avg_cadence_rpm,
    elevation_gain_m,
    intensity_type
FROM garmin_lakehouse.silver.splits
ORDER BY start_time_gmt DESC
LIMIT 20